# Speed Tests

In [1]:
# Standard libraries
import inspect
import time

In [2]:
# Third-party libraries
import mujoco
from rich.console import Console
from rich.traceback import install

In [3]:
# --- TERMINAL OUTPUT SETUP --- #
install(show_locals=False)
console = Console()
t = time.perf_counter()

In [4]:
# Local libraries
from ariel.body_phenotypes.robogen_lite.prebuilt_robots.gecko import gecko
from ariel.simulation import environments as envs
from ariel.simulation.environments import OlympicArena as World
from ariel.simulators import compile_model

msg = f"Imports completed in {time.perf_counter() - t:.2f} seconds."
console.log(msg)

[10:32:24] Imports completed in 1.81 seconds.                                                       ]8;id=3046286;file:///tmp/ipykernel_1417936/1199311418.py\1199311418.py]8;;\:]8;id=3046287;file:///tmp/ipykernel_1417936/1199311418.py#8\8]8;;\

## Load Pre-compiled

Each world class is built 1,000 times, with `load_precompiled` on and
off, and each class is timed on its own. `load_precompiled=True` reads
the world from `pre_compiled/<name>.xml` next to the world classes, a
cache that only `tools/compile_terrains.py` writes. No such file ships
with ARIEL, so with `True` every world falls back to building itself,
and both ways do the same work and take about the same time.

### Pre-compiled = True

In [5]:
iters = 1_000
precompiled_times: dict[str, float] = {}
for name, cls in inspect.getmembers(envs, inspect.isclass):
    t = time.perf_counter()
    for _ in range(iters):
        world = cls(load_precompiled=True)
    precompiled_times[name] = time.perf_counter() - t
    msg = f"Instantiated {name} {iters} times "
    msg += f"in {precompiled_times[name]:.2f} seconds."
    console.print(msg)
console.print(f"Total: {sum(precompiled_times.values()):.2f} seconds.")

Instantiated AmphitheatreTerrainWorld 1000 times in 4.59 seconds.

Instantiated BaseWorld 1000 times in 2.63 seconds.

Instantiated CompoundWorld 1000 times in 3.00 seconds.

Instantiated CraterTerrainWorld 1000 times in 4.45 seconds.

Instantiated OlympicArena 1000 times in 3.64 seconds.

Instantiated RuggedTerrainWorld 1000 times in 4.49 seconds.

Instantiated RuggedTiltedWorld 1000 times in 4.48 seconds.

Instantiated SimpleFlatWorld 1000 times in 2.70 seconds.

Instantiated SimpleTiltedWorld 1000 times in 2.81 seconds.

Total: 32.79 seconds.

### Pre-compiled = False

In [6]:
iters = 1_000
built_times: dict[str, float] = {}
for name, cls in inspect.getmembers(envs, inspect.isclass):
    t = time.perf_counter()
    for _ in range(iters):
        world = cls(load_precompiled=False)
    built_times[name] = time.perf_counter() - t
    msg = f"Instantiated {name} {iters} times "
    msg += f"in {built_times[name]:.2f} seconds."
    console.print(msg)
console.print(f"Total: {sum(built_times.values()):.2f} seconds.")

Instantiated AmphitheatreTerrainWorld 1000 times in 4.55 seconds.

Instantiated BaseWorld 1000 times in 2.57 seconds.

Instantiated CompoundWorld 1000 times in 2.93 seconds.

Instantiated CraterTerrainWorld 1000 times in 4.44 seconds.

Instantiated OlympicArena 1000 times in 3.59 seconds.

Instantiated RuggedTerrainWorld 1000 times in 4.43 seconds.

Instantiated RuggedTiltedWorld 1000 times in 4.46 seconds.

Instantiated SimpleFlatWorld 1000 times in 2.61 seconds.

Instantiated SimpleTiltedWorld 1000 times in 2.63 seconds.

Total: 32.20 seconds.

In [7]:
# Checks: every world class was timed both ways; none found a cached
# file, so both ways built the world and their totals are close
classes = dict(inspect.getmembers(envs, inspect.isclass))
assert set(precompiled_times) == set(built_times) == set(classes)
assert not any(
    cls(load_precompiled=True).is_precompiled for cls in classes.values()
)
ratio = sum(precompiled_times.values()) / sum(built_times.values())
same_work = (0.5, 2.0)  # loose: timings move with the machine's load
assert same_work[0] < ratio < same_work[1], ratio

## `world.spawn`

Each variant builds an `OlympicArena`, a gecko, and spawns the gecko
1,000 times. Correcting the spawn height costs a collision search, and
validating costs another; without the correction the gecko starts
half inside the floor.

In [8]:
def floor_contacts(world: World) -> int:
    """Count the contacts in a world at its first step.

    Returns
    -------
    int
        The number of contacts MuJoCo finds before the first step.
    """
    model = compile_model(world.spec)
    data = mujoco.MjData(model)
    mujoco.mj_forward(model, data)
    return int(data.ncon)


spawn_contacts: dict[str, int] = {}

### With Floor Collision Detection and Validation

In [9]:
spawn_times: dict[str, float] = {}
t = time.perf_counter()
iters = 1_000
for _ in range(iters):
    # Initialise world
    world = World()

    # Initialise robot body
    gecko_core = gecko()

    # Spawn robot in the world
    world.spawn(
        gecko_core.spec,
        position=(0, 0, 0),
        correct_collision_with_floor=True,
        validate_no_collisions=True,
    )

label = "With Floor Collision Detection and With Validation"
spawn_times[label] = time.perf_counter() - t
spawn_contacts[label] = floor_contacts(world)
msg = f"{label}: {iters} iterations in "
msg += f"{spawn_times[label]:.2f} seconds."
console.log(msg)

[10:33:37] With Floor Collision Detection and With Validation: 1000 iterations in 8.24 seconds.    ]8;id=3046294;file:///tmp/ipykernel_1417936/1709176658.py\1709176658.py]8;;\:]8;id=3046295;file:///tmp/ipykernel_1417936/1709176658.py#24\24]8;;\

### With Floor Collision Detection and Without Validation

In [10]:
t = time.perf_counter()
iters = 1_000
for _ in range(iters):
    # Initialise world
    world = World()

    # Initialise robot body
    gecko_core = gecko()

    # Spawn robot in the world
    world.spawn(
        gecko_core.spec,
        position=(0, 0, 0),
        correct_collision_with_floor=True,
        validate_no_collisions=False,
    )

label = "With Floor Collision Detection and Without Validation"
spawn_times[label] = time.perf_counter() - t
spawn_contacts[label] = floor_contacts(world)
msg = f"{label}: {iters} iterations in "
msg += f"{spawn_times[label]:.2f} seconds."
console.log(msg)

[10:33:44] With Floor Collision Detection and Without Validation: 1000 iterations in 7.12 seconds. ]8;id=3046302;file:///tmp/ipykernel_1417936/4103356422.py\4103356422.py]8;;\:]8;id=3046303;file:///tmp/ipykernel_1417936/4103356422.py#23\23]8;;\

### Without Floor Collision Detection and Without Validation

In [11]:
t = time.perf_counter()
iters = 1_000
for _ in range(iters):
    # Initialise world
    world = World()

    # Initialise robot body
    gecko_core = gecko()

    # Spawn robot in the world
    world.spawn(
        gecko_core.spec,
        position=(0, 0, 0),
        correct_collision_with_floor=False,
        validate_no_collisions=False,
    )

label = "Without Floor Collision Detection and Without Validation"
spawn_times[label] = time.perf_counter() - t
spawn_contacts[label] = floor_contacts(world)
msg = f"{label}: {iters} iterations in "
msg += f"{spawn_times[label]:.2f} seconds."
console.log(msg)

[10:33:50] Without Floor Collision Detection and Without Validation: 1000 iterations in 5.52       ]8;id=3046310;file:///tmp/ipykernel_1417936/3284921838.py\3284921838.py]8;;\:]8;id=3046311;file:///tmp/ipykernel_1417936/3284921838.py#23\23]8;;\
           seconds.                                                                                                

In [12]:
# Checks: all three variants ran; the corrected gecko touches nothing
# (it starts above the floor), the uncorrected one starts inside it
validated, corrected, uncorrected = spawn_times
assert set(spawn_contacts) == set(spawn_times)
assert spawn_contacts[validated] == spawn_contacts[corrected] == 0
assert spawn_contacts[uncorrected] > 0, spawn_contacts